# Week 5 — Build the Electrode Voltage Predictor app
**This notebook:**
1. Trains the **final app model** on all 2,634 electrodes: Week 2b features (composition + oxidation states) with the **E° reference shift**
2. Measures its honest accuracy (10 framework-based splits, and zero-shot for each ion) — shown in the app as the typical error
3. Writes all app files to `MyDrive/electrode_voltage_ml/app/` (ready for GitHub and Hugging Face Spaces)
4. Launches the app here with a **public shareable link** (valid for 72 hours)

The app takes only a **formula** (no crystal structure file), so it uses the chemistry model rather than the CHGNet one. No API key or GPU needed.

In [ ]:
!pip -q install gradio xgboost pymatgen
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"
import os
APP_DIR = f"{DATA_DIR}/app"
os.makedirs(APP_DIR, exist_ok=True)
print("App folder:", APP_DIR)

## 1. App code
These two files are the app. The featurizer is copied from the Week 2 and 2b notebooks so features are computed exactly as in training.

In [ ]:
%%writefile {APP_DIR}/featurize.py
"""
featurize.py — turns a host formula + working ion into the exact features used to train the model.
The code is copied from the Week 2 (composition) and Week 2b (oxidation-state) notebooks so that
the app computes features identically to training.
"""
import numpy as np
from pymatgen.core import Composition, Element
PROPS = {
    "X": lambda e: e.X,
    "radius": lambda e: float(e.atomic_radius) if e.atomic_radius is not None else np.nan,
    "Z": lambda e: e.Z,
    "mass": lambda e: float(e.atomic_mass),
    "row": lambda e: e.row,
    "group": lambda e: e.group,
    "mendeleev": lambda e: e.mendeleev_no,
    "ionization": lambda e: e.ionization_energy,
}
_cache = {}
def eprop(sym, name):
    key = (sym, name)
    if key not in _cache:
        try:
            v = PROPS[name](Element(sym)); _cache[key] = np.nan if v is None else float(v)
        except Exception:
            _cache[key] = np.nan
    return _cache[key]

TM = {str(e) for e in Element if e.is_transition_metal}
ANIONS = ["O", "F", "S", "Se", "N", "Cl", "Br"]
POLY = ["P", "Si", "B", "C", "Mo", "W"]

def host_features(formula):
    comp = Composition(formula).fractional_composition
    syms = [str(e) for e in comp.elements]; w = np.array([comp[e] for e in comp.elements])
    f = {}
    for name in PROPS:
        v = np.array([eprop(s, name) for s in syms])
        m = ~np.isnan(v)
        if m.any():
            ww = w[m] / w[m].sum(); vv = v[m]
            mean = (ww * vv).sum()
            f[f"host_{name}_mean"] = mean
            f[f"host_{name}_std"] = np.sqrt((ww * (vv - mean) ** 2).sum())
            f[f"host_{name}_min"], f[f"host_{name}_max"] = vv.min(), vv.max()
        else:
            for s in ("mean", "std", "min", "max"): f[f"host_{name}_{s}"] = np.nan
    f["host_n_elements"] = len(syms)
    f["host_frac_TM"] = sum(comp[e] for e in comp.elements if str(e) in TM)
    for a in ANIONS: f[f"host_frac_{a}"] = comp[a] if a in syms else 0.0
    has_O = "O" in syms
    f["host_polyanion"] = float(has_O and any(p in syms for p in POLY))
    tms = [s for s in syms if s in TM]
    f["host_TM_X_mean"] = np.mean([eprop(s, "X") for s in tms]) if tms else np.nan
    f["host_TM_Z_mean"] = np.mean([eprop(s, "Z") for s in tms]) if tms else np.nan
    return f

# --- Working-ion data -------------------------------------------------------
ION_CHARGE = {"Li": 1, "Na": 1, "K": 1, "Rb": 1, "Cs": 1, "Mg": 2, "Ca": 2, "Zn": 2, "Al": 3}
# Standard electrode potentials, V vs SHE (M^z+ + z e- -> M)
E0 = {"Li": -3.04, "Na": -2.71, "K": -2.93, "Rb": -2.98, "Cs": -3.03,
      "Mg": -2.37, "Ca": -2.87, "Zn": -0.76, "Al": -1.66}
TRAINED_IONS = ["Li", "Na", "K", "Mg", "Ca", "Zn"]

def ion_features(ion):
    e = Element(ion); q = ION_CHARGE[ion]
    return {"ion_charge": q, "ion_radius": float(e.ionic_radii.get(q, np.nan)),
            "ion_X": e.X, "ion_ionization": e.ionization_energy, "ion_mass": float(e.atomic_mass)}


# --- Oxidation-state features (Week 2b) -------------------------------------
ANION_Q = {"O": -2, "F": -1, "S": -2, "Se": -2, "Te": -2, "N": -3, "Cl": -1, "Br": -1, "I": -1}
# Elements that form positively charged centres of oxo-anions when oxygen is present
OXO_Q = {"P": 5, "Si": 4, "B": 3, "As": 5, "C": 4, "S": 6, "Se": 6, "N": 5, "Cl": 7, "I": 5, "Te": 6, "Ge": 4}
FIXED_Q = {"H": 1, "Li": 1, "Na": 1, "K": 1, "Rb": 1, "Cs": 1, "Be": 2, "Mg": 2, "Ca": 2, "Sr": 2, "Ba": 2,
           "Zn": 2, "Cd": 2, "Al": 3, "Ga": 3, "Sc": 3, "Y": 3, "La": 3, "Ce": 3, "Pr": 3, "Nd": 3,
           "Sm": 3, "Eu": 3, "Gd": 3, "Tb": 3, "Dy": 3, "Ho": 3, "Er": 3, "Tm": 3, "Yb": 3, "Lu": 3}
P_REDOX = {"Sb", "Sn", "Bi", "Pb", "Tl", "In"}
OX_RANGE = (-4.0, 8.0)          # physically sensible range; values outside -> unreliable

def _common_max(sym):
    try:
        cs = Element(sym).common_oxidation_states
        return max(cs) if cs else np.nan
    except Exception:
        return np.nan

def redox_state(formula):
    """Average oxidation state of redox-active metals by charge balance; NaN if unreliable."""
    comp = Composition(formula)
    syms = {str(e) for e in comp.elements}
    has_O = "O" in syms
    charge, redox = 0.0, {}
    for e in comp.elements:
        s, n = str(e), comp[e]
        if has_O and s in OXO_Q and s != "O":
            charge += OXO_Q[s] * n
        elif s in ANION_Q:
            charge += ANION_Q[s] * n
        elif s in FIXED_Q:
            charge += FIXED_Q[s] * n
        elif e.is_transition_metal or s in P_REDOX:
            redox[s] = n
        # anything else (elemental C, Si, Ge, P without O ...) contributes 0
    n_redox = sum(redox.values())
    if n_redox == 0:
        return np.nan, np.nan, redox
    avg = -charge / n_redox
    cmax = sum(_common_max(s) * n for s, n in redox.items()) / n_redox
    if not (OX_RANGE[0] <= avg <= OX_RANGE[1]):
        return np.nan, cmax, redox          # unphysical -> let the model treat it as missing
    return avg, cmax, redox

def ox_features(framework, discharged):
    comp = Composition(framework)
    syms = [str(e) for e in comp.elements]
    anion_free = not any(s in ANION_Q for s in syms)
    ox_c, cmax, redox = redox_state(framework)
    ox_d = redox_state(discharged)[0] if isinstance(discharged, str) else np.nan
    change = ox_c - ox_d if not (np.isnan(ox_c) or np.isnan(ox_d)) else np.nan
    if not np.isnan(change) and not (0 < change <= 4):
        change = np.nan                      # more than 4 electrons per metal is not physical
    excess = ox_c - cmax if not np.isnan(ox_c) else np.nan
    return {
        "ox_charged": ox_c,
        "ox_discharged": ox_d,
        "ox_change": change,
        "ox_common_max": cmax,
        "ox_excess": excess,
        "ox_above_max": float(excess > 0.2) if not np.isnan(excess) else 0.0,
        "ox_unreliable": float(len(redox) > 0 and np.isnan(ox_c)),
        "no_redox_metal": float(len(redox) == 0),
        "elemental_host": float(len(syms) == 1),
        "anion_free_host": float(anion_free),
    }

# --- Full feature vector ----------------------------------------------------
def discharged_formula(host, ion, x):
    """Host formula + x working ions per formula unit, e.g. ('FePO4', 'Li', 1) -> 'Li1FePO4'."""
    return f"{ion}{x:g}{Composition(host).formula.replace(' ', '')}"

def frac_ion(host, x):
    """Atomic fraction of the working ion in the discharged material (Materials Project 'fracA_discharge')."""
    n_host = Composition(host).num_atoms
    return x / (x + n_host)

def featurize(host, ion, x, columns):
    """Return one feature row (list) in the exact column order used for training."""
    f = {}
    f.update(host_features(host))
    f.update(ion_features(ion))
    f["frac_ion_discharge"] = frac_ion(host, x)
    f.update(ox_features(Composition(host).formula.replace(" ", ""), discharged_formula(host, ion, x)))
    return [f.get(c, np.nan) for c in columns], f


In [ ]:
%%writefile {APP_DIR}/app.py
"""
Electrode Voltage Predictor
---------------------------
Predicts the average voltage of an insertion-electrode host for a chosen working ion
(Li, Na, K, Mg, Ca, Zn, plus experimental Rb, Cs, Al) from its chemical formula.

Model: XGBoost trained on 2,634 DFT-computed electrodes from the Materials Project,
using composition + oxidation-state descriptors, with a physics-informed reference shift:
the model learns E_host = V + E°(ion) and converts back with the target ion's E°.
"""
import json
import os

import gradio as gr
import numpy as np
import pandas as pd
from pymatgen.core import Composition, Element
from xgboost import XGBRegressor

from featurize import E0, ION_CHARGE, TRAINED_IONS, discharged_formula, featurize

HERE = os.path.dirname(os.path.abspath(__file__))
META = json.load(open(os.path.join(HERE, "app_metadata.json")))
MODEL = XGBRegressor()
MODEL.load_model(os.path.join(HERE, META["model_file"]))
COLUMNS = META["feature_columns"]
REF_PATH = os.path.join(HERE, "reference_voltages.csv")
REF = pd.read_csv(REF_PATH) if os.path.exists(REF_PATH) else None
FARADAY_MAH = 26801.48          # mAh per mole of electrons

ION_CHOICES = [f"{i} (trained)" for i in TRAINED_IONS] + ["Rb (zero-shot)", "Cs (zero-shot)", "Al (experimental)"]


def _reduced(formula):
    try:
        return Composition(formula).reduced_formula
    except Exception:
        return None


def predict(host, ion_choice, x):
    host = (host or "").strip()
    ion = ion_choice.split()[0]
    if not host:
        return "Please enter a host formula, e.g. **FePO4** or **MnO2**."
    try:
        comp = Composition(host)
        _ = comp.num_atoms
        bad = [str(e) for e in comp.elements if not isinstance(e, Element)]
        if bad or comp.num_atoms == 0:
            raise ValueError
    except Exception:
        return f"Could not read **{host}** as a chemical formula. Use element symbols, e.g. `FePO4`, `Na2FeP2O7`."
    if x is None or x <= 0:
        return "The number of inserted ions per formula unit must be positive."
    if ion in [str(e) for e in comp.elements]:
        return (f"The host already contains **{ion}**. Enter the *charged* (ion-free) host, "
                f"e.g. `FePO4` instead of `LiFePO4`.")

    try:
        row, f = featurize(host, ion, float(x), COLUMNS)
        e_host = float(MODEL.predict(np.array([row], dtype=float))[0])
    except Exception as err:
        return f"Sorry, features could not be computed for **{host}** ({type(err).__name__}). Please check the formula."
    v = e_host - E0[ion]
    clipped = v < 0
    v = max(v, 0.0)

    dis = Composition(discharged_formula(host, ion, float(x)))
    capacity = float(x) * ION_CHARGE[ion] * FARADAY_MAH / dis.weight
    energy = v * capacity

    if ion in TRAINED_IONS:
        err = META["mae_in_distribution"]
        trust = f"Typical error for trained ions: **± {err:.2f} V** (mean absolute error on unseen host materials)."
    elif ion in ("Rb", "Cs"):
        err = META["mae_zero_shot_known_hosts"]
        trust = (f"⚠️ **{ion} was never in the training data.** The prediction relies on the E° reference shift "
                 f"(zero-shot). Typical zero-shot error: **± {err:.2f} V**, but {ion} is larger than any training ion, "
                 f"so treat it as an estimate.")
    else:
        trust = ("⚠️ **Experimental.** Al³⁺ is trivalent and no trivalent ion was in the training data. "
                 "Use only as a rough qualitative guide.")

    notes = []
    if f.get("no_redox_metal", 0) == 1 or f.get("elemental_host", 0) == 1:
        notes.append("This host has no transition metal for redox (e.g. graphite, Si, Sb). The model saw few such "
                     "anode-type hosts, so expect larger errors here.")
    if f.get("ox_unreliable", 0) == 1:
        notes.append("The oxidation states could not be balanced sensibly for this formula; the prediction is less reliable.")
    if f.get("ox_above_max", 0) == 1:
        notes.append("The redox metal in the charged host is beyond its common maximum oxidation state "
                     "(highly oxidised host; often high voltage or anion redox).")
    if clipped:
        notes.append("The raw prediction was below 0 V and has been set to 0 V.")

    ox_c, ox_d = f.get("ox_charged", np.nan), f.get("ox_discharged", np.nan)
    ox_line = (f"Average redox-metal oxidation state: **{ox_c:+.2f} → {ox_d:+.2f}** "
               f"(≈ {ox_c - ox_d:.2f} e⁻ per metal)") if not (np.isnan(ox_c) or np.isnan(ox_d)) else \
              "Average redox-metal oxidation state: not available for this host."

    ref_line = ""
    if REF is not None:
        red = _reduced(host)
        m = REF[(REF["framework_reduced"] == red) & (REF["working_ion"] == ion)]
        if len(m):
            vals = ", ".join(f"{vv:.2f} V" for vv in m["average_voltage"].head(3))
            ref_line = (f"\n\n📚 **This host is in the Materials Project data** for {ion}: DFT average voltage {vals}. "
                        f"(It may have been part of the training set.)")

    out = f"""## {v:.2f} V
**{comp.reduced_formula}** + {x:g} {ion}  →  **{dis.reduced_formula}**  (vs. {ion}/{ion}^{ION_CHARGE[ion]}+ metal)

| Property | Value |
|---|---|
| Predicted average voltage | **{v:.2f} V** |
| Theoretical gravimetric capacity | {capacity:.0f} mAh/g |
| Theoretical specific energy | {energy:.0f} Wh/kg |

{trust}

{ox_line}{ref_line}
"""
    if notes:
        out += "\n**Notes**\n" + "\n".join(f"- {n}" for n in notes)
    return out


ABOUT = f"""
### About this model
- **Data:** {META['n_training']:,} DFT-computed insertion electrodes (Li, Na, K, Mg, Ca, Zn) from the
  [Materials Project](https://materialsproject.org) (CC BY 4.0).
- **Features:** composition statistics of the host, working-ion properties and oxidation states from charge balance.
- **Physics-informed target:** the model learns the host potential *E = V + E°(ion)* and converts back with the
  ion's standard electrode potential. This keeps accuracy for known ions and roughly halves the error for
  ions never seen in training.
- **Accuracy:** MAE {META['mae_in_distribution']:.2f} V on unseen host materials (10 framework-based splits).
- **Limitations:** predictions are DFT-level estimates, not experimental voltages. Anode-type hosts
  (graphite, alloys), very high voltages (> 5 V) and rare chemistries are less reliable.

Author: Shaista Saddique · Code: see the GitHub repository linked in the README.
"""

with gr.Blocks(title="Electrode Voltage Predictor") as demo:
    gr.Markdown("# 🔋 Electrode Voltage Predictor\nPredict the average voltage of a battery electrode host "
                "for Li, Na, K, Mg, Ca or Zn insertion, from its chemical formula.")
    with gr.Row():
        with gr.Column(scale=1):
            host = gr.Textbox(label="Host formula (charged state, without the working ion)", value="FePO4",
                              placeholder="e.g. FePO4, MnO2, V2O5, Fe2(SO4)3")
            ion = gr.Dropdown(ION_CHOICES, value="Li (trained)", label="Working ion")
            x = gr.Number(value=1, label="Ions inserted per formula unit", minimum=0.01)
            btn = gr.Button("Predict voltage", variant="primary")
            gr.Examples([["FePO4", "Li (trained)", 1], ["FePO4", "Na (trained)", 1], ["MnO2", "Zn (trained)", 0.5],
                         ["V2O5", "Mg (trained)", 1], ["Fe2(SO4)3", "Na (trained)", 2], ["CoO2", "Li (trained)", 1]],
                        inputs=[host, ion, x])
        with gr.Column(scale=1):
            out = gr.Markdown()
    gr.Markdown(ABOUT)
    btn.click(predict, [host, ion, x], out)
    host.submit(predict, [host, ion, x], out)

if __name__ == "__main__":
    demo.launch()


## 2. Train the final model (with E° shift) and measure its accuracy

In [ ]:
import sys, json, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
sys.path.insert(0, APP_DIR)
from featurize import E0

feat = pd.read_csv(f"{DATA_DIR}/features_week2b.csv")
params = json.load(open(f"{DATA_DIR}/xgb_params.json"))["best_params"]
y = feat["average_voltage"].values
ions = feat["working_ion"].values
groups = feat["framework"].values
X = feat.drop(columns=["average_voltage", "working_ion", "framework"])
COLUMNS = list(X.columns)
shift = pd.Series(ions).map(E0).values
def new_model():
    return XGBRegressor(**params, min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1)

# honest in-distribution error: 10 framework-based splits
maes = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    p = new_model().fit(X.iloc[tr], (y + shift)[tr]).predict(X.iloc[te]) - shift[te]
    maes.append(mean_absolute_error(y[te], np.clip(p, 0, None)))
print(f"In-distribution MAE: {np.mean(maes):.3f} ± {np.std(maes):.3f} V")

# zero-shot error: leave each ion out (known hosts)
zs = {}
for t in ["Li", "Na", "K", "Mg", "Ca", "Zn"]:
    test = ions == t
    p = new_model().fit(X[~test], (y + shift)[~test]).predict(X[test]) - E0[t]
    zs[t] = mean_absolute_error(y[test], np.clip(p, 0, None))
print("Zero-shot MAE per ion:", {k: round(v, 3) for k, v in zs.items()})
print(f"Average zero-shot MAE: {np.mean(list(zs.values())):.3f} V")

In [ ]:
# final model on ALL data
model = new_model().fit(X, y + shift)
model.save_model(f"{APP_DIR}/voltage_model_e0.json")

meta = {"model_file": "voltage_model_e0.json", "feature_columns": COLUMNS,
        "n_training": int(len(y)), "training_ions": ["Li", "Na", "K", "Mg", "Ca", "Zn"],
        "mae_in_distribution": float(np.mean(maes)), "mae_in_distribution_std": float(np.std(maes)),
        "mae_zero_shot_known_hosts": float(np.mean(list(zs.values()))),
        "mae_zero_shot_per_ion": {k: float(v) for k, v in zs.items()},
        "target": "E_host = V + E0(ion); prediction V = E_host - E0(ion)", "xgb_params": params}
json.dump(meta, open(f"{APP_DIR}/app_metadata.json", "w"), indent=2)

# reference table: Materials Project DFT voltages (shown in the app when a host is already known)
from pymatgen.core import Composition
clean = pd.read_csv(f"{DATA_DIR}/electrodes_clean.csv")
ref = clean[["framework_formula", "working_ion", "average_voltage", "formula_discharge"]].copy()
ref["framework_reduced"] = ref["framework_formula"].apply(lambda f: Composition(f).reduced_formula)
ref.to_csv(f"{APP_DIR}/reference_voltages.csv", index=False)
print("Saved model, metadata and reference table to", APP_DIR)

## 3. Files for GitHub / Hugging Face Spaces

In [ ]:
import gradio
open(f"{APP_DIR}/requirements.txt", "w").write("gradio\nxgboost\npymatgen\nnumpy\npandas\n")
readme = f'''---
title: Electrode Voltage Predictor
emoji: 🔋
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: {gradio.__version__}
app_file: app.py
pinned: false
license: mit
---

# Electrode Voltage Predictor

Predicts the average voltage of a battery insertion-electrode host for Li, Na, K, Mg, Ca or Zn
from its chemical formula. XGBoost trained on {len(y):,} DFT-computed electrodes from the Materials Project,
with oxidation-state descriptors and a physics-informed reference shift (V + E°) that halves the error for
ions never seen in training.

- In-distribution MAE: {np.mean(maes):.2f} V (unseen host materials, 10 framework-based splits)
- Zero-shot MAE (ion excluded from training): {np.mean(list(zs.values())):.2f} V on average

Data: Materials Project (CC BY 4.0). Author: Shaista Saddique.
'''
open(f"{APP_DIR}/README.md", "w").write(readme)
print(sorted(os.listdir(APP_DIR)))

## 4. Launch the app
Click the **public URL** (ends in `gradio.live`) to open it in a new tab and share it. Stop the cell to shut the app down.

In [ ]:
os.chdir(APP_DIR)
import importlib, app as voltage_app
importlib.reload(voltage_app)
print(voltage_app.predict("FePO4", "Li (trained)", 1))
voltage_app.demo.launch(share=True, debug=False)